In [12]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from xgboost import XGBRegressor

In [13]:
df = pd.read_csv("data.csv")

In [14]:
features = df.drop(columns=[
    "date",
    "flood_risk",
    "landslide_risk",
    "wildfire_risk",
    "glacier_melt_risk"
])

targets = df[[
    "flood_risk",
    "landslide_risk",
    "wildfire_risk",
    "glacier_melt_risk"
]]


In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    features, targets, test_size=0.2, shuffle=False
)

In [16]:
model = MultiOutputRegressor(
    XGBRegressor(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.05
    )
)

In [17]:
model.fit(X_train, y_train)

print("✅ Model training complete")

# -----------------------------
# 6. Save model
# -----------------------------
joblib.dump(model, "xgb_multi_risk_model.pkl")

print("💾 Model saved as xgb_multi_risk_model.pkl")


✅ Model training complete
💾 Model saved as xgb_multi_risk_model.pkl


In [8]:
sample = X_test.iloc[0].values.reshape(1, -1)
pred = model.predict(sample)

print("\n🔍 Sample Prediction:")
print("Flood:", pred[0][0])
print("Landslide:", pred[0][1])
print("Wildfire:", pred[0][2])
print("Glacier Melt:", pred[0][3])


🔍 Sample Prediction:
Flood: 1.0
Landslide: 0.9999877
Wildfire: 0.85024744
Glacier Melt: 9.192816e-05


In [9]:
from sklearn.metrics import mean_squared_error

preds = model.predict(X_test)

print("\n📊 Evaluation:")
print("RMSE:", mean_squared_error(y_test, preds, squared=False))


📊 Evaluation:
RMSE: 0.03319649630686899


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/metrics/_regression.py:483: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [20]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score
import numpy as np

# Predictions
preds = model.predict(X_test)

In [21]:
rmse = np.sqrt(mean_squared_error(y_test, preds))
mae = mean_absolute_error(y_test, preds)

In [22]:
threshold = 0.6

pred_bin = (preds > threshold).astype(int)
y_test_bin = (y_test.values > threshold).astype(int)

accuracy = accuracy_score(
    y_test_bin.flatten(),
    pred_bin.flatten()
)


In [24]:
print("\nXGBOOST PERFORMANCE:")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"Accuracy: {accuracy:.2f}")


XGBOOST PERFORMANCE:
RMSE: 0.0546
MAE: 0.0258
Accuracy: 84.78
